### Flask Application for Nutrition Recommender App

In [ ]:
# Importing essential libraries
from flask import Flask, render_template, request, redirect, url_for, flash, session, redirect
import logging, re, os
from datetime import datetime
from werkzeug.security import generate_password_hash, check_password_hash 
import sqlite3
import csv
import pymongo
import math

# ------------------------------------
#Initialize Flask Application
app = Flask(__name__)
# ------------------------------------


# ------------------------------------
# In production set SECRET_KEY via environment variable
app.secret_key = os.environ.get("SECRET_KEY", "ftgongvsbn7283")

# ------------------------------------

# ------------------------------------
# Database path
DB_PATH = "patientdb.db"
#CSV_PATH = "pcos_data.csv"
# ------------------------------------

# ------------------------------------
def init_db():
   conn = sqlite3.connect(DB_PATH)
   conn.close()

# ------------------------------------
#Configure Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.FileHandler("app.log"), logging.StreamHandler()],
)
log = logging.getLogger(__name__)
# ------------------------------------


# ------------------------------------
# Simple in-memory storage
REGISTERED_USERS = [] # each item: {"username", "email", "age", "created_at"}
# ------------------------------------

# ------------------------------------
# Validation patterns

##Ensure that first name is only letters and hyphens
FIRSTNAME_PATTERN = re.compile(r'^[A-Za-z-]{1,50}$')

##Ensure that last name is only letters and hyphens
LASTNAME_PATTERN = re.compile(r'[A-Za-z-]{1,50}$')

## Ensures the email has a basic valid structure of name@domain.tld
EMAIL_PATTERN = re.compile(r'^[\w\.-]+@[\w\.-]+\.[A-Za-z]{2,}$')

## Strong password pattern that requires lowercase, uppercase, digit, special character, and minimum 8 characters.
PASSWORD_PATTERN = re.compile(r'^(?=.*[a-z])(?=.*[A-Z])(?=.*\d)(?=.*[@$!%*?&]).{8,}$')
# ------------------------------------

# ------------------------------------
# Routes for pages

# Route for the Home page
@app.route('/')
def home():
    return render_template('home.html')

# Route for the About page
@app.route('/about')
def about():
    return render_template('about.html')

# Route for the login page
@app.route('/login', methods=['GET', 'POST'])
def login():
    if request.method == "GET":
        return render_template("login.html")
    
    # ----- Post request handling -----
    email = request.form.get("email", "").strip()
    password = request.form.get("password", "")

    if not (email and password):
        flash("Please enter email and password.")
        return redirect(url_for("login"))

    # ----- Database connection and Query -----
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    #Querying the user table to find a user matching the email
    cursor.execute("""
        SELECT first_name, last_name, email, password
        FROM patientdata
        WHERE lower(email) = lower(?)
    """, (email,))
    row = cursor.fetchone()
    conn.close() #Close connection immediately after fetching data

    # ----- Check if we found a user in the database
    if not row:
        flash("Invalid email or password.")
        return redirect(url_for("login"))
    
    #Extracting the user data from the database row
    db_first_name, db_last_name, db_email, db_password = row

    #Verifying password
    if password != db_password:
        flash("Invalid email or password.")
        return redirect(url_for("login"))

    #Saving user information in session
    session["ID"] = db_email
    session["userFirstName"] = db_first_name

    return redirect(url_for("dashboard"))

# Route for the patient dashboard
@app.route("/dashboard")
def dashboard():
    if "ID" not in session:
        flash("Please log in to continue.")
        return redirect(url_for("login"))
    return render_template("dashboard.html")

# Route for the register page
@app.route('/register')
def register():
    return render_template('register.html')

if __name__ == "__main__":
    init_db()
    app.run(debug=False)  

 * Serving Flask app '__main__'
 * Debug mode: off


2026-07-08 16:36:51,551 [INFO] WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
2026-07-08 16:36:51,553 [INFO] Press CTRL+C to quit
2026-07-08 16:37:00,368 [INFO] 127.0.0.1 - - [08/Jul/2026 16:37:00] "GET / HTTP/1.1" 200 -
2026-07-08 16:37:00,610 [INFO] 127.0.0.1 - - [08/Jul/2026 16:37:00] "GET /static/images/woman_eating.jpg HTTP/1.1" 304 -
2026-07-08 16:37:00,618 [INFO] 127.0.0.1 - - [08/Jul/2026 16:37:00] "GET /static/css/home_style.css HTTP/1.1" 304 -
2026-07-08 16:37:03,667 [INFO] 127.0.0.1 - - [08/Jul/2026 16:37:03] "GET /login HTTP/1.1" 200 -
2026-07-08 16:37:03,724 [INFO] 127.0.0.1 - - [08/Jul/2026 16:37:03] "GET /static/css/login_style.css HTTP/1.1" 304 -
2026-07-08 16:37:14,385 [INFO] 127.0.0.1 - - [08/Jul/2026 16:37:14] "POST /login HTTP/1.1" 302 -
2026-07-08 16:37:14,405 [INFO] 127.0.0.1 - - [08/Jul/2026 16:37:14] "GET /dashboard HTTP/1.1" 200 -
2026-07-08 16:37:14,457 